# WHACS offshore waves at the goals

Reads WHACS (bulk and the four partitions: 0 = wind sea, 1-3 = swells) at the
WHACS node of every goal from the local copy of the global archive, cleans the
peak periods and writes the processed zarr used by the dynamic runs and the
reconstruction. The partitions are written as WHACS gives them: the energy they
miss is measured, and closed on the bulk, by the calibration (07b).

The archive is the `archive` of the WHACS `offshore_waves` entry: one folder per
variable with monthly files on the global SMC mesh, as CSIRO publishes them
([doi:10.25919/shdk-7p29](https://doi.org/10.25919/shdk-7p29)). Only the file
chunks that hold the goal nodes are read (`utils.whacs`).

**The archive is not part of HyGloW.** On the GeoOcean cluster it is
`/lustre/geocean/DATA/CSIRO/WHACS/global-smc`. Elsewhere, download the 20
variables read here (`hs`, `fp`, `dir`, `spr` and `phs`, `ptp`, `pdir`, `pspr`
0-3; about 14 TB for 1979-2026) or ask GeoOcean for the goal series of your case.

**Prerequisites:** `04_Select_Goals_by_Hindcasts.ipynb` (`<Case>_Goals.json`,
with a `whacs` node per goal).
**Next:** `07b_Satellite_Calibration.ipynb` validates (and calibrates) this
hindcast against satellite altimetry.

## Configuration


In [1]:
import xarray as xr

from utils import whacs
from utils.config import get_offshore_wave, resolve

offshore_name = "WHACS"  # offshore_waves entry with the archive
period = None  # months to read, e.g. slice("1979", "2023"); None = the whole archive
workers = 16  # processes reading the monthly files

archive = whacs.archive_path(offshore_name)
print(f"archive: {archive}")
print(f"output : {resolve(get_offshore_wave(offshore_name)['data_path'])}")

archive: /lustre/geocean/DATA/CSIRO/WHACS/global-smc
output : /lustre/geocean/WORK/users/javiert/projects/HyGloW/Netherlands/inputs/WHACS/processed/Netherlands_WHACS_goal.zarr


## Goal nodes

The archive seapoint of each goal's WHACS node (goals JSON `hindcast.whacs`);
`distance` (degrees) is ~0 when the node is a WHACS seapoint.

In [2]:
whacs.goal_seapoints(archive, offshore_name)

,seapoint,longitude,latitude,distance
goal,,,,
1,2323617,2.50000,51.50000,0.0
2,2334975,3.03125,52.03125,0.0
3,2324234,3.50000,52.50000,0.0
4,2324697,4.00000,53.00000,0.0
5,2325243,4.50000,53.50000,0.0
6,2326117,5.00000,54.00000,0.0
7,2326121,5.50000,54.00000,0.0
8,2326127,6.00000,54.00000,0.0
9,2326133,6.50000,54.00000,0.0


## Read the archive

Every variable of every month in `period` at the goal seapoints (`tp = 1 / fp`).

In [3]:
whacs_data = whacs.whacs_at_goals(offshore_name, period=period, workers=workers)
whacs_data

<xarray.Dataset> Size: 697MB
Dimensions:   (time: 412752, goal: 10)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2026-01-31T23:00:00
  * goal      (goal) int64 80B 1 2 3 4 5 6 7 8 9 10
    seapoint  (goal) int64 80B 2323617 2334975 2324234 ... 2326133 2326141
Data variables: (12/21)
    hs        (time, goal) float64 33MB 2.584 2.408 2.354 ... 1.578 1.54 1.506
    fp        (time, goal) float64 33MB 0.128 0.111 0.11 ... 0.17 0.177 0.19
    dir       (time, goal) float64 33MB 35.0 33.7 32.4 34.3 ... 95.4 96.5 99.2
    spr       (time, goal) float64 33MB 28.1 30.3 28.1 23.7 ... 36.1 34.6 32.6
    phs0      (time, goal) float64 33MB 2.584 2.408 2.354 ... 1.542 1.514 1.486
    ptp0      (time, goal) float64 33MB 7.81 9.02 9.05 8.8 ... 5.87 5.65 5.26
    ...        ...
    pspr2     (time, goal) float64 33MB nan nan nan nan ... 17.7 16.2 15.3 9.2
    phs3      (time, goal) float64 33MB nan nan nan nan ... 0.156 0.07 nan nan
    ptp3      (time, goal) float64 33MB nan nan nan nan ... 4.79 5.14 nan nan
    pdir3     (time, goal) float64 33MB nan nan nan nan ... 217.0 229.0 nan nan
    pspr3     (time, goal) float64 33MB nan nan nan nan nan ... 8.5 7.6 nan nan
    tp        (time, goal) float64 33MB 7.812 9.009 9.091 ... 5.882 5.65 5.263
Attributes: (12/54)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lat_max:              74.90625
    geospatial_lon_min:              -180.0
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_resolution:        P0DT1H0M0S
    hywaves_source:                  WHACS global archive /lustre/geocean/DAT...

Bulk `tp = 1 / fp` explodes when the spectral peak falls below the WHACS
frequency grid (hours with Tp of 50-250 s). Those take the peak period of the
most energetic partition (`clean_peak_periods`); partition periods above
`max_tp` become NaN.

In [4]:
from bluemath_tk.waves.hywaves.offshore import clean_peak_periods

max_tp = 25.0  # s, largest valid peak period
whacs_data = clean_peak_periods(whacs_data, max_tp=max_tp)
print(f"bulk tp replaced: {whacs_data.attrs['tp_replaced']}, partition tp removed: {whacs_data.attrs['ptp_removed']}")

bulk tp replaced: 10, partition tp removed: 0


## Write the zarr

In [5]:
whacs_data.to_zarr(resolve(get_offshore_wave(offshore_name)["data_path"]), mode="w")

In [6]:
xr.open_zarr(resolve(get_offshore_wave(offshore_name)["data_path"])).load()

<xarray.Dataset> Size: 697MB
Dimensions:   (time: 412752, goal: 10)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2026-01-31T23:00:00
  * goal      (goal) int64 80B 1 2 3 4 5 6 7 8 9 10
    seapoint  (goal) int64 80B 2323617 2334975 2324234 ... 2326133 2326141
Data variables: (12/21)
    dir       (time, goal) float64 33MB 35.0 33.7 32.4 34.3 ... 95.4 96.5 99.2
    fp        (time, goal) float64 33MB 0.128 0.111 0.11 ... 0.17 0.177 0.19
    hs        (time, goal) float64 33MB 2.584 2.408 2.354 ... 1.578 1.54 1.506
    pdir0     (time, goal) float64 33MB 35.0 33.7 32.4 34.3 ... 98.2 98.7 100.8
    pdir1     (time, goal) float64 33MB nan nan nan nan ... 19.5 14.4 7.6 332.8
    pdir2     (time, goal) float64 33MB nan nan nan nan ... 333.5 334.9 1.3
    ...        ...
    ptp0      (time, goal) float64 33MB 7.81 9.02 9.05 8.8 ... 5.87 5.65 5.26
    ptp1      (time, goal) float64 33MB nan nan nan nan ... 7.99 8.28 7.95 17.08
    ptp2      (time, goal) float64 33MB nan nan nan nan ... 17.25 17.16 8.06
    ptp3      (time, goal) float64 33MB nan nan nan nan ... 4.79 5.14 nan nan
    spr       (time, goal) float64 33MB 28.1 30.3 28.1 23.7 ... 36.1 34.6 32.6
    tp        (time, goal) float64 33MB 7.812 9.009 9.091 ... 5.882 5.65 5.263
Attributes: (12/56)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_resolution:        P0DT1H0M0S
    hywaves_source:                  WHACS global archive /lustre/geocean/DAT...
    tp_replaced:                     10
    ptp_removed:                     0